<a href="https://colab.research.google.com/github/pgordin/GraphsSN_2026_2/blob/main/Graphs3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Graph Functions**

In [2]:
import random
import networkx as nx
import numpy as np

random.seed(42)

In [3]:
def print_matrix(vertices, matrix):
  """
  Print a graph given by  the matrix.
  """
  n = len(matrix)
  if (vertices is not None) and (len(vertices) == n):
    vv = vertices
  else:
    vv = range(1, n+1)  ## numbers {1..n}
  for i in range(n):
    print(vv[i], ': ', end='', sep = '')
    for j in range(n):
      if matrix[i][j]:
        print(vv[j], end=' ')
    print()



In [4]:
def print_graph(graph):
  """
  Prints a graph given by an adjacency list (a dictionary).
  """
  for v in graph:
    print(v, ': ', end='', sep = '')
    for u in graph[v]:
      print(u, end=' ')
    print()

In [5]:
def add_vertex(graph, vertex):
  """
  adds a vertex to a graph
  """
  if vertex not in graph:
    graph[vertex] = []

In [32]:
def add_arc(graph, arc):
  """
  Adds an arc to a simple (di)graph.
  """
  u, v = arc
  add_vertex(graph, u)
  add_vertex(graph, v)
  if v not in graph[u]:
    graph[u].append(v)


In [6]:
def add_edge(graph, edge):
  """
  adds an edge to a graph (simple, undirected graph) (loops are not allowed)
  """
  u, v = edge
  add_vertex(graph, u)
  add_vertex(graph, v)
  if u == v:
    raise ValueError("loops are not allowed")
  if v not in graph[u]:
    graph[u].append(v)
  if u not in graph[v]:
    graph[v].append(u)

In [7]:
def random_graphn(n, p):
    graph = {}
    for i in range(1, n+1):
        add_vertex(graph, i)
        for j in range(1, i):
            if random.random() <= p:
                add_edge(graph, (i, j))
    return graph

In [8]:
def graph_to_matrix(graph):
    '''
    1. Take a list of vertices names
    2. Index them (e.g.: A: 0, B: 1)
    3. Create zero-matrix
    4. Using indexes from step 2 set values of connected vertices to 1
    '''
    n = len(graph)
    vertices = list(graph.keys())
    index = {v: i for i, v in enumerate(vertices)}
    matrix = [[0 for _ in range(n)] for _ in range(n)]
    for v in vertices:
        for neighbor in graph[v]:
            matrix[index[v]][index[neighbor]] = 1
    return vertices, matrix

In [9]:
def matrix_to_graph(vertices, matrix):
    graph = {v: [] for v in vertices}
    n = len(vertices)
    for i in range(n):
        for j in range(n):
            if matrix[i][j] == 1:
                graph[vertices[i]].append(vertices[j])
    return graph



def matrix_to_graph2(vertices, matrix):
    n = len(matrix)
    if (vertices is not None) and (len(vertices) == n):
      vv = vertices
    else:
      vv = range(1, n+1)  ## numbers {1..n}
    graph = {v: [] for v in vv}
    for i in range(n):
        for j in range(n):
            if matrix[i][j] == 1:
                graph[vv[i]].append(vv[j])
    return graph

In [24]:
def cycle(n):
    graph = {}
    for i in range(n):
        add_edge(graph, (i+1, (i+1) % n + 1))
    return graph

def cycle2(n):
    graph = {}
    for i in range(1, n):
        add_edge(graph, (i, i+1))
    add_edge(graph, (n, 1))
    return graph

**Testing**

In [11]:
matrix = [[0,0,1,0],
          [0,0,1,0],
          [1,1,0,0],
          [0,0,0,0]]

In [12]:
graph = {1: [3],
         2: [3],
         3: [1, 2],
         4: []}

In [13]:
vertices_letters = ['A', 'B', 'C', 'D']

In [14]:
print_graph(graph)

1: 3 
2: 3 
3: 1 2 
4: 


In [30]:
print_graph(cycle(5))

1: 2 5 
2: 1 3 
3: 2 4 
4: 3 5 
5: 4 1 


In [31]:
print_graph(cycle2(5))

1: 2 5 
2: 1 3 
3: 2 4 
4: 3 5 
5: 4 1 


In [16]:
print(graph_to_matrix(graph))

([1, 2, 3, 4], [[0, 0, 1, 0], [0, 0, 1, 0], [1, 1, 0, 0], [0, 0, 0, 0]])


In [17]:
print_matrix(vertices_letters,matrix)

A: C 
B: C 
C: A B 
D: 


In [29]:
print_graph(random_graphn(10, 0.3))

1: 3 6 
2: 4 5 
3: 1 8 9 
4: 2 6 8 9 
5: 2 7 8 10 
6: 1 4 7 9 10 
7: 5 6 8 9 
8: 3 4 5 7 
9: 3 4 6 7 
10: 5 6 


In [25]:
print(matrix_to_graph(vertices_letters, matrix))

{'A': ['C'], 'B': ['C'], 'C': ['A', 'B'], 'D': []}


In [26]:
print(matrix_to_graph2(None, matrix))

{1: [3], 2: [3], 3: [1, 2], 4: []}


# Reading and writing from/to files.

In [33]:
def graph_from_edges(filename, directed=0):
  """
  Reads a graph from a textfile. Each line of the file contains one word
  (a vertex name) or at leas two words (edge or arc name).
  Variable filename is a full path to the file.
  """
  graph = {}
  with open(filename, "r") as f:
    for line in f:
      words = line.strip().split()
      if len(words) == 1: # one word - a vertex description
        add_vertex(graph, words[0])
      elif len(words) >= 2: # at least two words - an edge description
        if directed:
          add_arc(graph, (words[0], words[1]))
        else:
          add_edge(graph, (words[0], words[1]))
  return graph

In [42]:
def graph_to_neighbourlist(graph, filename):
  """
  Writes a graph to a textfile. Each line of the file contains one word
  (a vertex name) or at leas two words (edge or arc name).
  Variable filename is a full path to the file.
  """
  with open(filename, "w") as f:
    for v in graph:
      line = f"{v}:"
      for u in graph[v]:
        line += f" {u}"
      line += "\n"
      f.write(line)

In [34]:
%%writefile edge.txt
a b
a c
b d
c e
f


Writing edge.txt


In [36]:
%cat edge.txt

a b
a c
b d
c e
f


In [37]:
print_graph(graph_from_edges("edge.txt"))

a: b c 
b: a d 
c: a e 
d: b 
e: c 
f: 


In [38]:
print_graph(graph_from_edges("edge.txt", directed=1))

a: b c 
b: d 
c: e 
d: 
e: 
f: 


In [39]:
!wget https://raw.githubusercontent.com/pgordin/GraphsSN_2026_2/refs/heads/main/weighted0.txt

--2026-11-02 11:02:22--  https://raw.githubusercontent.com/pgordin/GraphsSN_2026_2/refs/heads/main/weighted0.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 114 [text/plain]
Saving to: ‘weighted0.txt’

weighted0.txt       100%[===================>]     114  --.-KB/s    in 0s      

2026-11-02 11:02:23 (3.34 MB/s) - ‘weighted0.txt’ saved [114/114]



In [40]:
%cat weighted0.txt

A B 3
A E 10
B C 26
B D 12
C D 17
C F 13
C G 14
D E 7
D F 15
E F 8
E H 4
F G 9
F H 6
G H 16
G I 11


In [41]:
print_graph(graph_from_edges("weighted0.txt"))

A: B E 
B: A C D 
E: A D F H 
C: B D F G 
D: B C E F 
F: C D E G H 
G: C F H I 
H: E F G 
I: G 


In [43]:
graph_to_neighbourlist(graph_from_edges("weighted0.txt"), "graph.txt")

In [44]:
%cat graph.txt

A: B E
B: A C D
E: A D F H
C: B D F G
D: B C E F
F: C D E G H
G: C F H I
H: E F G
I: G
